In [1]:
import sqlite3
from pathlib import Path

DEFAULT_GAMES = ("卡卡頌", "花磚物語", "璀璨寶石")
current_user_id = "123abc"

# 專案文件建議從 repo 根目錄啟動 Jupyter；若直接在題目目錄啟動，
# 則退回使用目前目錄，確保資料庫仍與這份練習放在一起。
exercise_directory = Path("exercise/tibame/1-桌遊預約系統")
database_directory = exercise_directory if exercise_directory.is_dir() else Path.cwd()
DB_PATH = database_directory / "game_reservations.db"


def initialize_database(connection):
    """建立資料表，並只補上尚未存在的預設遊戲。"""
    # 字典初始化改為可重複執行的建表與 seed；不會清空既有預約。
    connection.execute(
        """
        CREATE TABLE IF NOT EXISTS game_reservation_states (
            id INTEGER PRIMARY KEY,
            game_name TEXT NOT NULL UNIQUE,
            reserved_by_user_id TEXT
        )
        """
    )
    connection.executemany(
        """
        INSERT INTO game_reservation_states (game_name)
        VALUES (?)
        ON CONFLICT(game_name) DO NOTHING
        """,
        ((game_name,) for game_name in DEFAULT_GAMES),
    )
    connection.commit()


def find_game_names(connection, message):
    """找出出現在使用者訊息中的遊戲名稱。"""
    rows = connection.execute(
        """
        SELECT game_name
        FROM game_reservation_states
        ORDER BY id
        """
    ).fetchall()
    matched_games = [
        row["game_name"]
        for row in rows
        if row["game_name"] in message
    ]  # fmt: skip
    return matched_games


def end_service():
    """服務結束"""
    print("服務結束")


def show_database_error(connection, error):
    """回復失敗的資料庫操作並顯示錯誤。"""
    connection.rollback()
    print(f"資料庫操作失敗：{error}")


def cancel_reservations(connection):
    """取消目前使用者的所有預約。"""
    try:
        rows = connection.execute(
            """
            SELECT game_name
            FROM game_reservation_states
            WHERE reserved_by_user_id = ?
            ORDER BY id
            """,
            (current_user_id,),
        ).fetchall()
        canceled_games = [row["game_name"] for row in rows]

        if len(canceled_games) == 0:
            print("你目前沒有預約。")
            return

        # 字典逐項設為 None，改成一次 UPDATE 並以一次 commit 完成。
        connection.execute(
            """
            UPDATE game_reservation_states
            SET reserved_by_user_id = NULL
            WHERE reserved_by_user_id = ?
            """,
            (current_user_id,),
        )
        connection.commit()

        game_text = "、".join(canceled_games)
        print(f"已取消預約：{game_text}")
    except sqlite3.Error as error:
        show_database_error(connection, error)


def show_available_games(connection):
    """顯示所有尚未被預約的遊戲。"""
    try:
        # 字典篩選 None，改成使用 WHERE ... IS NULL 查詢。
        rows = connection.execute(
            """
            SELECT game_name
            FROM game_reservation_states
            WHERE reserved_by_user_id IS NULL
            ORDER BY id
            """
        ).fetchall()
        available_games = [row["game_name"] for row in rows]

        if len(available_games) == 0:
            print("目前沒有可預約的遊戲。")
        else:
            game_text = "\n".join(available_games)
            print(f"目前可預約的遊戲：\n{game_text}")
    except sqlite3.Error as error:
        show_database_error(connection, error)


def show_unknown_command():
    """無法理解指令"""
    print("我不明白這個指令。")
    print("你可以查看遊戲、預約遊戲、取消預約，或輸入「結束」。")


def reserve_game(connection, message):
    """從訊息中找出遊戲，然後為目前使用者預約。"""
    try:
        matched_games = find_game_names(connection, message)

        if len(matched_games) == 0:
            print("找不到遊戲名稱，請說明想預約哪一款遊戲。")
            return

        if len(matched_games) > 1:
            game_text = "、".join(matched_games)
            print(f"你提到了多款遊戲：{game_text}，請一次指定一款。")
            return

        game_name = matched_games[0]

        # 字典指定使用者，改成只在尚未預約時才成功的原子 UPDATE。
        cursor = connection.execute(
            """
            UPDATE game_reservation_states
            SET reserved_by_user_id = ?
            WHERE game_name = ?
              AND reserved_by_user_id IS NULL
            """,
            (current_user_id, game_name),
        )

        if cursor.rowcount == 1:
            connection.commit()
            print(f"已為你預約：{game_name}")
            return

        row = connection.execute(
            """
            SELECT reserved_by_user_id
            FROM game_reservation_states
            WHERE game_name = ?
            """,
            (game_name,),
        ).fetchone()
        connection.commit()

        if row is None:
            print("找不到遊戲名稱，請說明想預約哪一款遊戲。")
        elif row["reserved_by_user_id"] == current_user_id:
            print(f"你已經預約過：{game_name}")
        else:
            print(f"{game_name} 已被其他使用者預約。")
    except sqlite3.Error as error:
        show_database_error(connection, error)


def main():
    connection = None

    try:
        connection = sqlite3.connect(DB_PATH)
        connection.row_factory = sqlite3.Row
        initialize_database(connection)
    except sqlite3.Error as error:
        if connection is not None:
            connection.close()
        print(f"資料庫初始化失敗：{error}")
        return

    try:
        while True:
            msg = input("輸入訊息: ").strip()

            if msg == "結束":
                end_service()
                break
            elif "取消" in msg:
                cancel_reservations(connection)
            elif "預約" in msg or "訂" in msg:
                reserve_game(connection, msg)
            elif "遊戲" in msg or "有哪些" in msg:
                show_available_games(connection)
            else:
                show_unknown_command()
    finally:
        # 交易在各操作完成時 commit；服務結束時關閉長連線。
        connection.close()


main()

找不到遊戲名稱，請說明想預約哪一款遊戲。
已為你預約：花磚物語
我不明白這個指令。
你可以查看遊戲、預約遊戲、取消預約，或輸入「結束」。
我不明白這個指令。
你可以查看遊戲、預約遊戲、取消預約，或輸入「結束」。
我不明白這個指令。
你可以查看遊戲、預約遊戲、取消預約，或輸入「結束」。
我不明白這個指令。
你可以查看遊戲、預約遊戲、取消預約，或輸入「結束」。
已取消預約：花磚物語
你目前沒有預約。
我不明白這個指令。
你可以查看遊戲、預約遊戲、取消預約，或輸入「結束」。
我不明白這個指令。
你可以查看遊戲、預約遊戲、取消預約，或輸入「結束」。
我不明白這個指令。
你可以查看遊戲、預約遊戲、取消預約，或輸入「結束」。
我不明白這個指令。
你可以查看遊戲、預約遊戲、取消預約，或輸入「結束」。
已為你預約：花磚物語
已取消預約：花磚物語
你目前沒有預約。
已為你預約：花磚物語
已取消預約：花磚物語
已為你預約：花磚物語
服務結束
